<a href="https://colab.research.google.com/github/NMH1203/DL2026-13-18/blob/main/Notebooks/NDN/test_best.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q ultralytics==8.4.173 pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 11.3 MB/s eta 0:00:00


In [3]:
!git clone https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git /content/Dataset

Cloning into '/content/Dataset'...
remote: Enumerating objects: 24082, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 24082 (delta 2), reused 1 (delta 1), pack-reused 24069 (from 3)
Receiving objects: 100% (24082/24082), 768.64 MiB | 37.82 MiB/s, done.
Resolving deltas: 100% (123/123), done.
Updating files: 100% (32693/32693), done.


In [4]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Searching for required files/folders...\n")

targets = [
    "exdark_yolo_clahe_only",
    "exdark_yolo_clahe_bilateral",
    "clahe_yolov8n",
    "clahe_bilateral_yolov8n",
]

for target in targets:
    matches = list(DRIVE_ROOT.rglob(target))
    print(f"{target}:")
    for path in matches[:10]:
        print("  ", path)
    print()

Searching for required files/folders...

exdark_yolo_clahe_only:
   /content/drive/MyDrive/exdark_yolo_clahe_only

exdark_yolo_clahe_bilateral:
   /content/drive/MyDrive/exdark_yolo_clahe_bilateral

clahe_yolov8n:
   /content/drive/MyDrive/clahe_yolov8n

clahe_bilateral_yolov8n:
   /content/drive/MyDrive/clahe_bilateral_yolov8n



In [5]:
from pathlib import Path
import yaml

CLASSES = {
    0: "Bicycle",
    1: "Boat",
    2: "Bottle",
    3: "Bus",
    4: "Cat",
    5: "Cup",
    6: "Motorbike",
    7: "People",
    8: "Table",
    9: "car",
    10: "chair",
    11: "dog",
}

CLAHE_ONLY_DATASET = Path(
    "/content/drive/MyDrive/exdark_yolo_clahe_only"
)

CLAHE_BILATERAL_DATASET = Path(
    "/content/drive/MyDrive/exdark_yolo_clahe_bilateral"
)

EVAL_DIR = Path("/content/evaluation")
EVAL_DIR.mkdir(exist_ok=True)

clahe_only_yaml = {
    "path": str(CLAHE_ONLY_DATASET),
    "train": "test/images",
    "val": "test/images",
    "test": "test/images",
    "nc": 12,
    "names": CLASSES,
}

clahe_bilateral_yaml = {
    "path": str(CLAHE_BILATERAL_DATASET),
    "train": "test/images",
    "val": "test/images",
    "test": "test/images",
    "nc": 12,
    "names": CLASSES,
}

with open(EVAL_DIR / "clahe_only.yaml", "w") as f:
    yaml.safe_dump(clahe_only_yaml, f, sort_keys=False)

with open(EVAL_DIR / "clahe_bilateral.yaml", "w") as f:
    yaml.safe_dump(clahe_bilateral_yaml, f, sort_keys=False)

print("Evaluation YAML files created.")
print(EVAL_DIR / "clahe_only.yaml")
print(EVAL_DIR / "clahe_bilateral.yaml")

Evaluation YAML files created.
/content/evaluation/clahe_only.yaml
/content/evaluation/clahe_bilateral.yaml


In [6]:
%cd /content/Dataset

!python src/LBN/train_ninh.py evaluate \
    --data /content/evaluation/clahe_only.yaml \
    --weights /content/drive/MyDrive/clahe_yolov8n/weights/best.pt \
    --split test \
    --device 0 \
    --project /content/drive/MyDrive/Project18_Test_Results \
    --name clahe_yolov8n_test \
    --batch 16 \
    --imgsz 640

/content/Dataset
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.1 ms, read: 0.6±0.3 MB/s, size: 493.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1UoOhiCFe_w8UYr8iaLMltPb-6fXjdTMQ/exdark_yolo_clahe_only/test/labels... 734 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 734/734 1.0it/s 11:58
WARNING ⚠️ val: Failed to save cache to /content/drive/.shortcut-targets-by-id/1UoOhiCFe

In [7]:
%cd /content/Dataset

!python src/LBN/train_ninh.py evaluate \
    --data /content/evaluation/clahe_bilateral.yaml \
    --weights /content/drive/MyDrive/clahe_bilateral_yolov8n/weights/best.pt \
    --split test \
    --device 0 \
    --project /content/drive/MyDrive/Project18_Test_Results \
    --name clahe_bilateral_yolov8n_test \
    --batch 16 \
    --imgsz 640

/content/Dataset
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.1 ms, read: 0.5±0.2 MB/s, size: 408.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1TtH7OiNnu0wUOu4kqa0YKbrc8obvXWOR/exdark_yolo_clahe_bilateral/test/labels... 734 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 734/734 1.2it/s 10:07
WARNING ⚠️ val: Failed to save cache to /content/drive/.shortcut-targets-by-id/1TtH7OiNnu0wUOu4kqa0YKbrc8obvXWOR/exdark_yolo_clahe_bilateral/test/labels.cache: [Errno 30] Read-only file system: '/content/drive/.shortcut-targets-by-id/1TtH7OiNnu0wUOu4kqa0YKbrc8obvXWOR/exdark_yolo_clahe_bilateral/test/labels.cache'
                 Class     Images  Instances      Box(P      